# Fine-tune Llama 3.2 1B for MathInstruct

This notebook fine-tunes `unsloth/Llama-3.2-1B-Instruct-bnb-4bit` on `TIGER-Lab/MathInstruct` with Unsloth LoRA adapters, `TrainingArguments`, and early stopping while keeping the math-tutor prompt style.

In [ ]:
%pip install torch==2.10.0 --index-url https://download.pytorch.org/whl/cu130
%pip install unsloth trl datasets huggingface_hub peft

In [1]:
import torch
torch.__version__

'2.10.0+cu130'

In [2]:
import os
from pathlib import Path
import unsloth
from datasets import DatasetDict, load_dataset
from huggingface_hub import login
from transformers import DataCollatorForLanguageModeling, EarlyStoppingCallback, TrainingArguments, set_seed
from trl.trainer.sft_trainer import SFTTrainer
from unsloth import FastLanguageModel

set_seed(42)

MODEL_NAME = "unsloth/Llama-3.2-1B-Instruct-bnb-4bit"
DATASET_NAME = "TIGER-Lab/MathInstruct"
WORKDIR = Path.cwd().resolve()
OUTPUT_DIR = WORKDIR / "artifacts" / "mathinstruct-lora-llama32-1b"
MAX_SEQ_LENGTH = 2048
MAX_EVAL_SAMPLES = 100
TRAIN_FRACTION = 0.03
EARLY_STOPPING_PATIENCE = 2
EARLY_STOPPING_THRESHOLD = 0.0
USE_BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
SYSTEM_PROMPT = (
    "You are a helpful math tutor. "
    "Solve the problem with clear reasoning and end with a concise final answer."
)

HF_TOKEN = os.getenv("HF_TOKEN")
if HF_TOKEN:
    login(token=HF_TOKEN, add_to_git_credential=False)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")
print(f"BF16 available: {USE_BF16}")
print(f"Unsloth version: {getattr(unsloth, '__version__', 'unknown')}")
print(f"Artifacts directory: {OUTPUT_DIR}")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


Unable to import `torchao` Tensor objects. This may affect loading checkpoints serialized with `torchao`


Device: cuda
BF16 available: True
Unsloth version: 2025.11.1
Artifacts directory: /home/thuannd/Repository/aio-llmops/notebooks/artifacts/mathinstruct-lora-llama32-1b


In [3]:
raw_train = load_dataset(DATASET_NAME, split="train")
split_dataset = raw_train.train_test_split(test_size=MAX_EVAL_SAMPLES, seed=42)

train_base = split_dataset["train"].shuffle(seed=42)
train_count = max(1, int(len(train_base) * TRAIN_FRACTION))
dataset = DatasetDict(
    {"train": train_base.select(range(train_count)),
     "val": split_dataset["test"]}
)

print(dataset)
print(f"Train fraction: {TRAIN_FRACTION} -> {len(dataset['train'])} samples")
sample = dataset["train"][20]
print("\nInstruction preview:\n")
print(sample["instruction"])
print("\nReference solution preview:\n")
print(sample["output"])

DatasetDict({
    train: Dataset({
        features: ['source', 'output', 'instruction'],
        num_rows: 7858
    })
    val: Dataset({
        features: ['source', 'output', 'instruction'],
        num_rows: 100
    })
})
Train fraction: 0.03 -> 7858 samples

Instruction preview:

In how many different ways can the letters of the word "FAME" be rearrangement?
Answer Choices: (A) 26 (B) 24 (C) 28 (D) 12 (E) 30

Reference solution preview:

Option 'B'
The total number of arrangements is
4P4 = 4! = 24


In [4]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    dtype=torch.bfloat16,
 )

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "up_proj",
        "down_proj",
        "o_proj",
        "gate_proj",
    ],
    use_rslora=True,
    use_gradient_checkpointing="unsloth",
    random_state=42,
 )
model.config.use_cache = False
model.print_trainable_parameters()

==((====))==  Unsloth 2025.11.1: Fast Llama patching. Transformers: 4.57.2.
   \\   /|    NVIDIA GeForce RTX 3090. Num GPUs = 1. Max memory: 23.555 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu130. CUDA: 8.6. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Unsloth 2025.11.1 patched 16 layers with 16 QKV layers, 16 O layers and 16 MLP layers.


trainable params: 11,272,192 || all params: 1,247,086,592 || trainable%: 0.9039


In [5]:
def build_messages(problem, answer=None):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": problem.strip()},
    ]
    if answer is not None:
        messages.append({"role": "assistant", "content": answer.strip()})
    return messages

def render_conversation(problem, answer=None, add_generation_prompt=False):
    messages = build_messages(problem, answer)
    if getattr(tokenizer, "chat_template", None):
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=add_generation_prompt,
        )

    prompt = (
        f"{SYSTEM_PROMPT}\n\n"
        f"Problem:\n{problem.strip()}\n\n"
        "Solution:\n"
    )
    if answer is None:
        return prompt
    return f"{prompt}{answer.strip()}{tokenizer.eos_token}"

def format_batch(batch):
    texts = [
        render_conversation(problem, answer)
        for problem, answer in zip(batch["instruction"], batch["output"])
    ]
    return {"text": texts}

formatted_dataset = dataset.map(
    format_batch,
    batched=True,
    remove_columns=dataset["train"].column_names,
 )

def tokenize_batch(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_SEQ_LENGTH)

tokenized_dataset = formatted_dataset.map(
    tokenize_batch,
    batched=True,
    remove_columns=["text"],
 )

print(formatted_dataset)

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 7858
    })
    val: Dataset({
        features: ['text'],
        num_rows: 100
    })
})


In [6]:
print(render_conversation(dataset["train"][0]["instruction"], dataset["train"][0]["output"])[:800])

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 22 Apr 2026

You are a helpful math tutor. Solve the problem with clear reasoning and end with a concise final answer.<|eot_id|><|start_header_id|>user<|end_header_id|>

If two different solutions of alcohol with a respective proportion of water to alcohol of 3:1 and 2:3 were combined, what is the concentration of alcohol in the new solution if the original solutions were mixed in equal amounts?
Answer Choices: (A) 30.0% (B) 36.6% (C) 42.5% (D) 44.4% (E) 60.0%<|eot_id|><|start_header_id|>assistant<|end_header_id|>

Let's think about the multi-choice question step by step.
Alcohol: Water
First mixture -- 3:1 -- 15:5
Second mixture -- 2:3 -- 8:12
Equal amounts of the mixture were co


In [7]:
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    num_train_epochs=5,
    learning_rate=2e-4,
    weight_decay=0.01,
    warmup_steps=200,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    lr_scheduler_type="cosine",
    report_to="none",
    bf16=USE_BF16,
    fp16=torch.cuda.is_available() and not USE_BF16,
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    remove_unused_columns=False,
    seed=42,
 )

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["val"],
    data_collator=data_collator,
    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=EARLY_STOPPING_PATIENCE,
            early_stopping_threshold=EARLY_STOPPING_THRESHOLD,
        )
    ],
 )

train_result = trainer.train()
train_result.metrics

The model is already on multiple devices. Skipping the move to device specified in `args`.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 7,858 | Num Epochs = 5 | Total steps = 2,460
O^O/ \_/ \    Batch size per device = 8 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (8 x 2 x 1) = 16
 "-____-"     Trainable parameters = 11,272,192 of 1,247,086,592 (0.90% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,Validation Loss
100,0.877200,0.927699
200,0.793200,0.881248
300,0.820000,0.863019
400,0.730400,0.855236
500,0.709000,0.843805
600,0.645800,0.844333
700,0.656500,0.836137
800,0.631700,0.829156
900,0.637700,0.822159
1000,0.475700,0.847535


Unsloth: Not an error, but LlamaForCausalLM does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient


{'train_runtime': 1507.2469,
 'train_samples_per_second': 26.067,
 'train_steps_per_second': 1.632,
 'total_flos': 6.332167179674419e+16,
 'train_loss': 0.7261128703030673,
 'epoch': 2.236012207527976}

In [8]:
eval_metrics = trainer.evaluate()
eval_metrics

{'eval_loss': 0.8574861288070679,
 'eval_runtime': 3.0119,
 'eval_samples_per_second': 33.201,
 'eval_steps_per_second': 4.316,
 'epoch': 2.236012207527976}

In [9]:
FastLanguageModel.for_inference(model)

problem = dataset["val"][0]["instruction"]
reference_solution = dataset["val"][0]["output"]
prompt = render_conversation(problem, answer=None, add_generation_prompt=True)
model_device = next(model.parameters()).device
inputs = tokenizer(prompt, return_tensors="pt").to(model_device)

with torch.inference_mode():
    generated = model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=True,
        temperature=0.2,
        top_p=0.9,
        repetition_penalty=1.1,
    )

generated_tokens = generated[0][inputs["input_ids"].shape[1]:]
prediction = tokenizer.decode(generated_tokens, skip_special_tokens=True)

print("Problem:\n")
print(problem)
print("\nModel solution:\n")
print(prediction)
print("\nReference solution:\n")
print(reference_solution)

Problem:

In how many different Q ways can 4 ladies and 4 gentlemen be seated at a round table so that all ladies sit together?
Answer Choices: (A) 70 (B) 288 (C) 576 (D) 10,080 (E) 20,160

Model solution:

Let's think about the multi-choice question.
The number of arrangements of n distinct objects in a row is given by n!
The number of circular permutations of n distinct objects is given by (n-1)!
So, for this problem we have:
Number of ways to seat the ladies = 4! = 24
Number of ways to seat the gentlemen = 4! = 24
Total number of ways to seat both = 24*24 = 576
The answer is C.

Reference solution:

Treat the 4 ladies as one object, now you have 5 objects to arrange around a table (m1,m2,m3,m4,women). This can be done in (5-1)! ways
And there are 4! ways to arrange ladies among themselves
Answer Q= (4!)^2 = 576 or C


In [10]:
final_adapter_dir = OUTPUT_DIR / "final_adapter"
final_adapter_dir.mkdir(parents=True, exist_ok=True)

trainer.model.save_pretrained(final_adapter_dir)
tokenizer.save_pretrained(final_adapter_dir)

print(f"Saved LoRA adapter to: {final_adapter_dir.resolve()}")

Saved LoRA adapter to: /home/thuannd/Repository/aio-llmops/notebooks/artifacts/mathinstruct-lora-llama32-1b/final_adapter


In [ ]:
model.push_to_hub(
    "VLAI-AIVN/Llama-3.2-1B-Instruct-mathqa-lora",
    commit_message="Add model ckpt",
)

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Saved model to https://huggingface.co/thuanan/Llama-3.2-1B-Instruct-mathqa-lora


In [ ]:
tokenizer.push_to_hub(
    "VLAI-AIVN/Llama-3.2-1B-Instruct-mathqa-lora",
    commit_message="Add tokenizer ckpt",
)

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            